# Gather Full Text

This notebook gathers the articles' full text using the PMC API

In [1]:
from datasets import load_dataset
import pandas as pd
from tqdm import tqdm
import pickle
import os

In [ ]:
# Note that this dataset requires trust_remote_code=True, which is only available for datasets<4.0.0

pmc_ds = load_dataset("pmc/open_access", split="train", trust_remote_code=True)

In [ ]:
pmc_ds[0]

In [2]:
df_references = pd.read_parquet("../data/00_references/merged_references.parquet.brotli")

In [3]:
# For the mapping to work, we need a dictionary from PMID to full_text.
# This dict is saved in memory as it takes a long time to build - it 
# requires iterating over the entire PMC dataset.

mapping_path = "../src/dataset_utils/pmid_to_full_text.pkl"
pmids_needed = set(df_references["PMID"].astype(str).tolist())

if os.path.exists(mapping_path):
    print(f"Loading PMID → full_text mapping from {mapping_path}")
    with open(mapping_path, "rb") as f:
        pmid_to_full_text = pickle.load(f)
else:
    print(f"Mapping file not found. Building PMID → full_text mapping...")
    pmid_to_full_text = {
        row["pmid"]: row["text"]
        for row in tqdm(pmc_ds, total=len(pmc_ds), desc="Building PMID → full_text mapping")
        if row["pmid"] in pmids_needed
    }
    with open(mapping_path, "wb") as f:
        pickle.dump(pmid_to_full_text, f)
    print(f"Mapping saved to {mapping_path}")

Loading PMID → full_text mapping from ../src/dataset_utils/pmid_to_full_text.pkl


In [4]:
df_references["full_text"] = df_references["PMID"].astype(str).map(pmid_to_full_text)

In [5]:
# We only keep the rows that are available in PMC.
print(f"Number of references before filtering for full text: {df_references.shape[0]}")
df_references = df_references[df_references["full_text"].notnull()].reset_index(drop=True)
print(f"Number of references after filtering for full text: {df_references.shape[0]}")

Number of references before filtering for full text: 6658
Number of references after filtering for full text: 5251


In [6]:
df_references.to_parquet("../data/01_full_text/1_references_with_full_text.parquet.brotli", index=False, compression="brotli")